In [ ]:
# If running in Google Colab, before beginning, run this cell to install optimization packages
!pip install pyomo
!apt-get install -y -qq glpk-utils

# Introduction to Pyomo

In this module, we looked at a few different ways of formulated linear optimization models - including ways to transform nonlinear models into linear models. But once we have a linear model, how can we find an solution? In this notebook, we'll introduce a Python package called Pyomo that let's us solve linear optimization models. We will discuss some of the algorithms that are used behind the scenes and delve deeper into Pyomo's capabilities in a later lesson.

`Pyomo` allows us to formulate mathematical optimization problems conveniently in Python, and passes these problems to a solver backend. `Pyomo` can use many solvers: `glpk` is a free and convenient solver backend for small and medium-sized optimization problems.


Let's return to our model of transit ridership: we want to maximize the total ridership (the objective function) by offering a certain number of trips for heavy rail, light rail, bus rapid transit (BRT) and bus (our decision variables $x_1, x_2, x_3, x_4$ respectively), subject to a constraint that the total cost of all trips falls within our budget. This gave us the following model:


**Maximize Total Ridership:**

$
400x_1 + 125x_2 + 60x_3 + 40x_4
$

**Subject to: costs being less than or equal to budget**

$
 20x_1 + 8x_2 + 4x_3 + 3x_4 \leq 500
$

**and:  number of trips being non-negative:**

$
x_1, x_2, x_3, x_4 \geq 0
$


Below is an implementation of this model in Pyomo. We will spend more time discussing the syntax of Pyomo in a later class, which will give you the tools to formulate your own models. For now, the main thing to note is that we can solve these models using Python. Let's note a few things in the code:

1) We have defined our model decision variables - for example $x_1$ is `model.x1`. The `domain=NonNegativeReals` instruction takes care of the nonnegativity constraints for us.

2) We have defined the objective function as `Objective(
    expr=400 * model.x1 + 125 * model.x2 + 60 * model.x3 + 40 * model.x4,
    sense=maximize
)` - note how this corresponds to the mathematical objective function in our model above. `sense=maximize` tells Pyomo we are maximizing the function.

3) We also define our constraint similarly: `Constraint(
    expr=20 * model.x1 + 8 * model.x2 + 4 * model.x3 + 3 * model.x4 <= 500
)`

With all this in mind, try solving the model by running the cell and viewing the output.

In [ ]:
#Import Pyomo library
from pyomo.environ import *

# Create a Pyomo model
model = ConcreteModel()

#Define decision variables
model.x1 = Var(domain=NonNegativeReals)
model.x2 = Var(domain=NonNegativeReals)
model.x3 = Var(domain=NonNegativeReals)
model.x4 = Var(domain=NonNegativeReals)

#Define the objective function:
model.objective = Objective(
    expr=400 * model.x1 + 125 * model.x2 + 60 * model.x3 + 40 * model.x4,
    sense=maximize
)

#Define the constraint: 20x1 + 8x2 + 4x3 + 3x4 <= 500
model.constraint = Constraint(
    expr=20 * model.x1 + 8 * model.x2 + 4 * model.x3 + 3 * model.x4 <= 500
)


#Solve the model
solver = SolverFactory('glpk')
result = solver.solve(model)

# Print the optimal solution
print("Status:", result.solver.status)
print("Termination Condition:", result.solver.termination_condition)
print("\nOptimal Solution:")
print(f"Heavy Rail x1 = {model.x1.value:.2f}")
print(f"Light Rail x2 = {model.x2.value:.2f}")
print(f"BRT x3 = {model.x3.value:.2f}")
print(f"Bus x4 = {model.x4.value:.2f}")
print(f"Optimal Total Ridership= {model.objective():.2f}")

Once it has solved, we have a solution to the transit ridership problem: we see that the model was solved to optimality, and that the optimal ridership is 10000, acheived by putting 25 trips in heavy rail, and nowhere else.

Now, lets add in the two additional constraints we looked at for this model, $x_2 \geq 10$ and $60x_3 \geq 300$, labeled `ADDITIONAL CONSTRAINT` below, and resolve by running the cell.

In [ ]:
# Create a Pyomo model
model = ConcreteModel()

#Define decision variables
model.x1 = Var(domain=NonNegativeReals)
model.x2 = Var(domain=NonNegativeReals)
model.x3 = Var(domain=NonNegativeReals)
model.x4 = Var(domain=NonNegativeReals)

#Define the objective function:
model.objective = Objective(
    expr=400 * model.x1 + 125 * model.x2 + 60 * model.x3 + 40 * model.x4,
    sense=maximize
)

#Define the constraint: 20x1 + 8x2 + 4x3 + 3x4 <= 500
model.constraint1 = Constraint(
    expr=20 * model.x1 + 8 * model.x2 + 4 * model.x3 + 3 * model.x4 <= 500
)

#ADDITIONAL CONSTRAINT
#Define the constraint: x2 >= 10
model.constraint2 = Constraint(
    expr= model.x2 >= 10
)

#ADDITIONAL CONSTRAINT
#Define the constraint 60x3 >= 300
model.constraint3 = Constraint(
    expr= 60*model.x3 >= 300
)

#Solve the model
solver = SolverFactory('glpk')
result = solver.solve(model)

# Print the optimal solution
print("Status:", result.solver.status)
print("Termination Condition:", result.solver.termination_condition)
print("\nOptimal Solution:")
print(f"Heavy Rail x1 = {model.x1.value:.2f}")
print(f"Light Rail x2 = {model.x2.value:.2f}")
print(f"BRT x3 = {model.x3.value:.2f}")
print(f"Bus x4 = {model.x4.value:.2f}")
print(f"Optimal Total Ridership= {model.objective():.2f}")

We see that the new constraints have unsurprisingly caused some reallocation of service to Light rail and BRT, for a slight reduction in optimal total ridership (from 10000 to 9500). This is not surprising - we have added additional constraints on the decision variables that were not there before, so the objective function will at best not improve, and possibly (as in this case) decrease.

Now let's try our linearized model from the previous notebook, which we developed to eliminate an absolute value in the objective function:

*Minimize*:  
$ 5X + 2V $

*Subject* to:  
$V \geq Y$

$V \geq -Y$

$ X + Y \geq 9 $

To put this in Pyomo, we follow essentially the same approach as above, with a few differences to note: in this case, we will allow $Y$ to take on positive or negative values by specifying `domain=Reals` - we'll let the sign be determined by our constraints. Additionally, this is a minimization problem, so we specify `sense=minimize` in the objective function. Otherwise, the setup is very similar. Try solving the model by running the cell below:

In [ ]:
# Create a Pyomo model
model = ConcreteModel()

# Define decision variables
model.X = Var(domain=NonNegativeReals)  # X >= 0
model.Y = Var(domain=Reals)            # Y is unrestricted (we'll enforce constraints)
model.V = Var(domain=NonNegativeReals) # V >= 0

# Define the objective function
model.objective = Objective(
    expr=5 * model.X + 2 * model.V,
    sense=minimize
)

# Define the constraints
# Constraint 1: V >= Y
model.constraint1 = Constraint(
    expr=model.V >= model.Y
)

# Constraint 2: V >= -Y
model.constraint2 = Constraint(
    expr=model.V >= -model.Y
)

# Constraint 3: X + Y >= 9
model.constraint3 = Constraint(
    expr=model.X + model.Y >= 9
)

# Solve the model
solver = SolverFactory('glpk')  # Using the GLPK solver (ensure it's installed)
result = solver.solve(model)

# Display the results
print("Solver Status:", result.solver.status)
print("Termination Condition:", result.solver.termination_condition)
print("\nOptimal Solution:")
print(f"X = {model.X.value:.2f}")
print(f"Y = {model.Y.value:.2f}")
print(f"V = {model.V.value:.2f}")
print(f"Optimal Objective Value (Z) = {model.objective():.2f}")